# 02 · Índice de riesgo, clustering y MLOps
### Riesgo de acceso a la salud por municipio en Colombia · Fase 2

Este notebook ejecuta el **pipeline completo** (`riesgo_salud.pipeline.ejecutar`, lo
mismo que `make pipeline`) y muestra sus resultados:

1. **Preparación:** extracción, limpieza, homologación DIVIPOLA y validaciones (notebook 01).
2. **Índice de riesgo de acceso (0-100)**, con prueba de sensibilidad de los pesos y comparación con pesos de PCA.
3. **Clustering:** 15 modelos (K-means, jerárquico y mezcla gaussiana con k = 3 a 7), cada uno registrado en **MLflow**.
4. **Registro del modelo** en el *Model Registry* de MLflow. El alias `produccion` solo se mueve si el modelo pasa la regla de selección.
5. **Monitoreo** frente al corte anterior (PSI y cambios de nivel).
6. **Salidas para Power BI** y un resumen automático en `resultados/<fecha>/`.

> Todo el código está en `src/riesgo_salud/`; los parámetros (EPS intervenidas, pesos del
> índice, modelos a comparar, umbrales) en `config/config.yaml`. Este notebook solo llama
> al pipeline y muestra los resultados.

In [ ]:
import json
from datetime import date

import mlflow
import pandas as pd
from IPython.display import Image, Markdown, display

from riesgo_salud.config import cargar_config, ultimo_corte
from riesgo_salud.pipeline import ejecutar
from riesgo_salud.tracking import uri_tracking

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

cfg = cargar_config()
FECHA = ultimo_corte(cfg) or date.today().isoformat()  # o fije una fecha: FECHA = "2026-10-25"
print("Carpeta del proyecto:", cfg["rutas"]["base"])
print("Corte:", FECHA)
print("EPS intervenidas:", cfg["eps_intervenidas"]["lista"])
print("Pesos del índice:", {k: v["peso"] for k, v in cfg["indice"]["componentes"].items()})

## 1. Ejecutar el pipeline completo
Si el corte ya está en `data/raw/<fecha>/` no se vuelve a descargar. Con los datos reales
tarda menos de un minuto; la mayor parte es la prueba de estabilidad de los 15 modelos.

In [ ]:
r = ejecutar(cfg, FECHA)
SALIDA = r["salida"]

## 2. Calidad de los datos
Cada chequeo crítico debe estar en `True`. Si alguno falla, el pipeline se detiene antes de entrenar.

In [ ]:
display(pd.DataFrame(json.loads((SALIDA / "calidad.json").read_text(encoding="utf-8"))))
display(r["cruce"])

## 3. Índice de riesgo de acceso
El índice combina cinco componentes en percentiles: dependencia de EPS intervenidas (35 %),
oferta de IPS (30 %, invertida), concentración de EPS (15 %), ruralidad (10 %) y régimen
subsidiado (10 %). Los niveles se definen por cuantiles: 30 % Bajo, 40 % Medio, 20 % Alto y
10 % Muy alto.

In [ ]:
t = r["tabla"]
display(
    t.groupby("nivel_riesgo")
    .agg(municipios=("cod_mpio", "size"), afiliados=("afiliados", "sum"))
    .reindex(cfg["indice"]["etiquetas_niveles"])
)
display(Image(SALIDA / "figuras" / "04_distribucion_indice.png"))

In [ ]:
print("Robustez del índice:")
for k, v in r["metricas_indice"].items():
    print(f"  {k:<28} {v:.3f}")
print("\nPesos que propone PCA:", json.loads((SALIDA / "metricas_indice.json").read_text())["pesos_pca"])

> **Resultado del corte 2026-10-29.** Al mover los pesos al azar 500 veces, el ranking casi no
> cambia (Spearman mediano 0,983; 0,937 en el 5 % de los peores casos): **los pesos
> elegidos son estables alrededor de su valor**.
>
> Pero los pesos de **PCA cuentan otra historia** (Spearman 0,63 y solo 26 % de
> coincidencia en el 10 % de mayor riesgo). PCA le da poco peso a la dependencia (10 %)
> y a la oferta (5 %), porque el primer componente captura sobre todo la ruralidad, el
> régimen y la concentración, que se mueven juntos. Esto no invalida el índice: PCA mide
> qué variables varían juntas, no cuáles importan para el acceso. Pero es una decisión que
> hay que **justificar ante el profesor**: elegimos pesos por criterio (dependencia y
> oferta son el núcleo del problema) y no por varianza.

In [ ]:
cols = [
    "ranking_riesgo",
    "dpto",
    "nom_mpio",
    "afiliados",
    "indice_riesgo",
    "nivel_riesgo",
    "pct_intervenidas",
    "ips_x_10mil",
    "hhi",
    "eps_principal",
    "prob_top10",
]
display(t.nsmallest(20, "ranking_riesgo")[cols])

> **Resultado del corte 2026-10-29.** Los primeros puestos son municipios pequeños con el 100 % de
> afiliados en EPS intervenidas y entre 0,7 y 2 sedes de IPS por 10.000 afiliados: Argelia
> (Cauca), Leiva (Nariño), Mogotes (Santander), Florencia (Cauca) y Solita (Caquetá).
> Asmet Salud es la EPS principal en 8 de los 15 primeros. `prob_top10` dice en qué
> proporción de las 500 simulaciones de pesos el municipio quedó en el 10 % de mayor riesgo.

## 4. Comparación de modelos de clustering
Regla de selección (`modelado.seleccionar`):
1. Ningún grupo con menos del 5 % de los municipios.
2. Estabilidad (ARI en 20 muestras bootstrap) de al menos 0,70.
3. Entre los que cumplen, el de mayor silhouette.

Si ningún candidato cumple, se describe el mejor balance, pero **no** se promueve a producción.

In [ ]:
display(r["comparacion"].sort_values("silhouette", ascending=False))
print(
    "Modelo elegido:",
    r["elegido"]["algoritmo"],
    "con k =",
    int(r["elegido"]["k"]),
    "· cumple la regla:",
    bool(r["elegido"]["cumple_regla"]),
)
display(Image(SALIDA / "figuras" / "01_seleccion_modelo.png"))

> **Resultado del corte 2026-10-29.** Solo tres candidatos cumplen la regla: K-means con k = 3, 4 y 7.
> Gana **K-means con k = 3** (silhouette 0,236, estabilidad 0,82). El jerárquico y la mezcla
> gaussiana son inestables (ARI entre 0,31 y 0,59). Un silhouette de 0,24 indica que los
> grupos **se solapan**: los municipios forman un continuo más que grupos separados. Por eso
> el índice es la herramienta principal para priorizar y los grupos sirven para
> describir perfiles.

## 5. Perfiles de los grupos
El grupo 1 es el de mayor riesgo promedio. El mapa de calor muestra cuántas desviaciones
estándar se aleja cada grupo del promedio nacional en cada variable.

In [ ]:
display(
    r["perfiles"][
        [
            "grupo",
            "municipios",
            "afiliados",
            "indice_promedio",
            "pct_intervenidas",
            "hhi",
            "ips_x_10mil",
            "pct_rural",
            "pct_subsidiado",
            "pct_nivel_muy_alto",
            "descripcion",
        ]
    ]
)
display(Image(SALIDA / "figuras" / "02_perfiles_grupos.png"))

> **Resultado del corte 2026-10-29.** 
>
> | Grupo | Municipios | Afiliados | Índice promedio | Rasgos |
> |---|---|---|---|---|
> | 1 | 446 | 5,3 M | 63,5 | alta dependencia de EPS intervenidas, afiliación muy concentrada, muy rural |
> | 2 | 388 | 6,0 M | 43,3 | baja dependencia de EPS intervenidas, muy subsidiado |
> | 3 | 288 | 38,9 M | 38,2 | más contributivo, más urbano, afiliación más diversificada |
>
> El grupo 1 es el foco: 446 municipios rurales y concentrados en EPS intervenidas. El grupo 2
> es igual de subsidiado pero depende de EPS no intervenidas (allí está buena parte de
> Antioquia con Savia Salud): si la lista de intervenidas cambia, es el grupo que más se moverá.

## 6. Mapa de niveles de riesgo

In [ ]:
import plotly.express as px

kw = dict(
    lat="latitud",
    lon="longitud",
    color="nivel_riesgo",
    size="afiliados",
    size_max=26,
    hover_name="nom_mpio",
    hover_data={
        "dpto": True,
        "indice_riesgo": ":.1f",
        "grupo": True,
        "eps_principal": True,
        "latitud": False,
        "longitud": False,
    },
    category_orders={"nivel_riesgo": cfg["indice"]["etiquetas_niveles"]},
    color_discrete_sequence=["#fdd0a2", "#fd8d3c", "#d94801", "#7f2704"],
    zoom=4.3,
    center={"lat": 4.6, "lon": -74.1},
    height=650,
    title="Nivel de riesgo de acceso a la salud por municipio",
)
try:
    fig = px.scatter_map(t, map_style="carto-positron", **kw)
except AttributeError:
    fig = px.scatter_mapbox(t, mapbox_style="carto-positron", **kw)
fig.show()
display(pd.read_csv(SALIDA / "resumen_departamentos.csv").head(15))

## 7. MLflow: experimentos y modelo registrado
Cada uno de los 15 modelos quedó como un *run* hijo dentro del *run* del pipeline. El
modelo elegido quedó en el **Model Registry** con su firma (*signature*) y, si cumplió la
regla, con el alias `produccion`. Cualquiera puede cargarlo por su alias y usarlo con datos nuevos.

Para ver la interfaz de MLflow, en una terminal aparte: `make mlflow` y abrir
http://127.0.0.1:5001.

In [ ]:
mlflow.set_tracking_uri(uri_tracking(cfg))
runs = mlflow.search_runs(experiment_names=[cfg["mlflow"]["experimento"]])
cols = [
    c
    for c in runs.columns
    if c.startswith(
        (
            "tags.mlflow.runName",
            "params.algoritmo",
            "params.k",
            "metrics.silhouette",
            "metrics.estabilidad_ari",
            "metrics.davies_bouldin",
        )
    )
]
display(
    runs[cols].dropna(subset=["params.algoritmo"]).sort_values("metrics.silhouette", ascending=False).head(15)
)

nombre, alias = cfg["mlflow"]["modelo_registrado"], cfg["mlflow"]["alias_produccion"]
version = mlflow.MlflowClient().get_model_version_by_alias(nombre, alias)
modelo = mlflow.sklearn.load_model(f"models:/{nombre}@{alias}")
X = t[cfg["clustering"]["variables"]]
print(f"{nombre}@{alias} = versión {version.version}")
print(
    "Coincidencia del modelo registrado con los grupos asignados:", (modelo.predict(X) == t["grupo"]).mean()
)

## 8. Monitoreo entre cortes
En el primer corte no hay con qué comparar. Cuando haya un corte nuevo (por ejemplo, el
del mes siguiente con `make pipeline`), el pipeline lo compara con el anterior: PSI por
variable, municipios que cambian de nivel y de grupo, y concordancia entre el modelo
anterior y el nuevo. Con `REVISAR_HOY = True` se descarga el corte de hoy y se compara.

In [ ]:
REVISAR_HOY = False
if REVISAR_HOY and date.today().isoformat() != FECHA:
    r2 = ejecutar(cfg, date.today().isoformat())
    display(Markdown((r2["salida"] / "monitoreo.md").read_text(encoding="utf-8")))
else:
    display(Markdown((SALIDA / "monitoreo.md").read_text(encoding="utf-8")))

## 9. Resumen automático y archivos para Power BI
`municipios_riesgo.csv` es la tabla para Power BI: una fila por municipio con los
indicadores, el índice, el nivel y el grupo (ver `docs/guia_powerbi.md`). Las pruebas
automáticas se corren con `make test` (y GitHub Actions las corre en cada cambio).

In [ ]:
display(Markdown((SALIDA / "resumen.md").read_text(encoding="utf-8")))
print("Archivos generados:")
for p in sorted(SALIDA.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(SALIDA))

## 10. Conclusiones (corte 2026-10-29)

1. **El índice prioriza 113 municipios en nivel "Muy alto"** (1,4 millones de afiliados),
   sobre todo de Santander, Boyacá, Caquetá, Cauca y Valle del Cauca. En la mitad de ellos
   casi toda la población está en EPS intervenidas (Nueva EPS, Coosalud y Asmet Salud son la
   EPS principal en 106 de los 113) y hay 1,5 sedes de IPS o menos por 10.000 afiliados.
2. **Los pesos son estables, pero son una decisión del equipo.** Perturbarlos casi no
   cambia el ranking; PCA propone otros pesos porque mide covariación, no importancia.
3. **Tres perfiles de municipios** con K-means (k = 3, estabilidad 0,82). El silhouette de
   0,24 dice que los grupos se solapan: sirven para describir, el índice para priorizar.
4. **El flujo es reproducible y monitoreado:** dependencias exactas (`uv.lock`), datos
   versionados por corte, contratos de calidad que detienen el pipeline, 15 runs en
   MLflow con promoción controlada y monitoreo PSI mes a mes.